# Team Transformers · Final Kaggle submissions

The scores below are from the final team report. The competition provides the hidden labels to its evaluator. This notebook verifies both final checkpoints against their submitted files.

In [1]:
import hashlib
import json
import os
from pathlib import Path
import torch
from predict_compressed import load_student, reproduce, WEIGHTS, REFERENCE
from competition_data import sha256
from student import parameter_counts

results = json.loads(Path("results/submissions.json").read_text())
for key in ("compressed", "teammate_soup"):
    entry = results[key]
    print(f"{entry['filename']}   public {entry['public']:.3f}   private {entry['private']:.3f}")

tm_compress_seed51_ema_final_flip.csv   public 0.906   private 0.875
submission_soup_kd3_tta10.csv   public 0.905   private 0.867


## Load and count the final EMA

The compressed model uses one student at inference. Its checkpoint includes the saved BatchNorm statistics.

In [2]:
model, normalization = load_student()
assert sha256(WEIGHTS) == results["compressed"]["weights_sha256"]
print(f"Trainable parameters: {parameter_counts(model)['trainable']:,}")
print("Weights SHA-256:", sha256(WEIGHTS))
history = json.loads(Path("results/history.json").read_text())
print(f"Completed epochs: {len(history)}")
print(f"Final teacher agreement on the unlabeled test images: {history[-1]['test_agree_ema']:.2%}")

Trainable parameters: 4,918,602
Weights SHA-256: e70fec47366499b633d8e447cea745064e9671b32940951766c0c257b80b5d21
Completed epochs: 40
Final teacher agreement on the unlabeled test images: 99.70%


## Reproduce the submitted CSV

Provide the extracted competition files in data/ or set CIFAR10_DATA. The check decodes the original 32×32 PNGs and applies CPU FP32 original/flip probability averaging.

In [3]:
data = Path(os.environ.get("CIFAR10_DATA", "data"))
if (data / "test.csv").is_file():
    torch.set_num_threads(4)
    generated = reproduce(data)
    assert generated == REFERENCE.read_bytes()
    print("2,000 of 2,000 predictions match the submitted CSV in sample order.")
    print("CSV SHA-256:", hashlib.sha256(generated).hexdigest())
else:
    print("Set the data directory to run the image verification.")

2,000 of 2,000 predictions match the submitted CSV in sample order.
CSV SHA-256: 56ddeeebeeab7ad9e16c45e2d21874cde2e434ace8354a394b4d60ab066a3aab


## Reproduce the KD3 soup

Load the round-3 SAM student soup and average its ten inference views on the same test images.

In [4]:
from model import load_model, predict as soup_probabilities, count_params
from competition_data import read_images

soup_record = results["teammate_soup"]
soup_path = Path(soup_record["checkpoint"])
assert sha256(soup_path) == soup_record["weights_sha256"]
soup = load_model(str(soup_path), device="cuda" if torch.cuda.is_available() else "cpu")
print(f"Trainable parameters: {count_params(soup):,}")
if (data / "test.csv").is_file():
    ids, images = read_images(data, "test")
    labels = soup_probabilities(soup, images, views=10).argmax(1)
    text = "id,label\n" + "".join(f"{i},{label}\n" for i, label in zip(ids, labels, strict=True))
    out = text.encode("ascii")
    assert out == (Path("results") / soup_record["filename"]).read_bytes()
    assert hashlib.sha256(out).hexdigest() == soup_record["csv_sha256"]
    print("2,000 of 2,000 predictions match the submitted KD3 soup CSV in sample order.")
else:
    print("Set the data directory to run the image verification.")

Trainable parameters: 4,918,602
2,000 of 2,000 predictions match the submitted KD3 soup CSV in sample order.
